# Lab 02 · Notebook 02 — A Feed-Forward Network for MNIST Digits

In **Notebook 01** we classified iris flowers from 4 hand-measured numbers. Now we take the same
6-step recipe and point it at something much bigger: **70 000 handwritten digit images**.

The model is still a plain **feed-forward neural network** (FNN, also called a *multi-layer
perceptron* or MLP) — just `nn.Linear` layers with ReLU between them. We will **not** use
convolutional layers here; those come later in the course. Doing MNIST "the FNN way" first is
deliberate: it works surprisingly well (~98% accuracy), and it makes very clear *what* an FNN
cannot do with images — which is exactly the motivation you will need later.

## Same recipe, bigger data

| # | Step | Notebook 01 (IRIS) | **Notebook 02 (MNIST)** |
|---|------|--------------------|-------------------------|
| 1 | Load & preprocess | 150 rows from `sklearn` | **70 000 images** from `torchvision`, `ToTensor` + `Normalize` |
| 2 | Split | `train_test_split` ×2 | **official test set** + `random_split` for validation |
| 3 | Model | 4 → 10 → 10 → 3 (193 params) | **784 → 256 → 128 → 10** (235 146 params) + Dropout |
| 4 | Loss & optimizer | `CrossEntropyLoss` + Adam | same (the recipe does not change!) |
| 5 | Train | **full batch**, 1 update/epoch | **mini-batches** via `DataLoader`, 860 updates/epoch |
| 6 | Evaluate & visualise | accuracy, curves, 3×3 matrix | accuracy, per-class accuracy, 10×10 matrix, error gallery |

## What is genuinely new in this notebook

1. **Images as input** — and the `flatten` step that squashes a 28×28 picture into a 784-long
   vector so a `Linear` layer can accept it.
2. **`Dataset` and `DataLoader`** — PyTorch's standard data pipeline: batching, shuffling and
   loading, all handled for you.
3. **Mini-batch gradient descent** — many small weight updates per epoch instead of one big one.
   This is what fixes the underfitting we saw in Notebook 01.
4. **GPU support** — the `device` pattern (`.to(device)`) that every real PyTorch script uses.
5. **Dropout** — our first regularisation technique, and the reason `model.train()` /
   `model.eval()` finally *matter*.

## The problem in one picture

```
  one 28x28 grayscale image          flatten            hidden layers          10 output scores
  ┌───────────────────────┐       ┌────────────┐                              ┌──────────────┐
  │  ░░░░▓▓▓▓░░░░         │       │  784       │   [256]      [128]           │ score "0"    │
  │  ░░▓▓░░░░▓▓░░         │  ──▶  │  numbers   │──▶ +ReLU  ──▶ +ReLU  ──▶     │ score "1"    │
  │  ░░░░░░░░▓▓░░         │       │  in a row  │   +Drop      +Drop           │   ...        │
  │  ░░░░░░▓▓░░░░         │       └────────────┘                              │ score "9"    │
  │  ░░░░▓▓░░░░░░         │                                                   └──────────────┘
  │  ░░▓▓▓▓▓▓▓▓░░         │        784 = 28 x 28                                largest score
  └───────────────────────┘                                                     = the prediction
       label: 7
```

## What you should be able to do after this notebook

* Explain why an image must be **flattened** before a `Linear` layer, and what information that
  destroys.
* Build a `Dataset` → `DataLoader` pipeline and say what `batch_size` and `shuffle` do.
* Explain why mini-batch training converges much faster **per epoch** than full-batch training.
* Move a model and its data to the **GPU** correctly.
* Explain what Dropout does during training vs during evaluation.
* Read a 10×10 confusion matrix and name which digit pairs a model confuses.

> **Runtime:** training takes roughly **3–6 minutes on a Colab CPU** and **under 2 minutes on a
> GPU**. You can switch on a GPU via *Runtime → Change runtime type → T4 GPU*, but the CPU is
> perfectly fine for a network this size — the code below detects whichever you have. Run the
> cells **in order**, top to bottom.

---
# Step 1 · Load and Preprocess the MNIST Data

**Goal:** get 70 000 images into memory as normalised tensors, and look at them before we train on
them.

### About MNIST

The most famous benchmark in machine learning: handwritten digits collected from US Census Bureau
employees and high-school students, published by LeCun, Cortes and Burges in 1998.

| Property | Value |
|---|---|
| Images | 70 000 total — **60 000 train** + **10 000 test** (an *official*, fixed split) |
| Size | 28 × 28 pixels |
| Channels | 1 (grayscale) |
| Pixel values | integers 0–255, where 0 = black background and 255 = white ink |
| Classes | 10, the digits `0`–`9` |
| Balance | roughly 10% per class (not exactly equal, but close) |

Because MNIST ships with an **official test set**, we do *not* need `train_test_split` for it — we
just have to carve a validation set out of the 60 000 training images. Using the official test set
is what makes results comparable across textbooks and papers.

### 1.1 Imports, reproducibility, and the device

Three habits worth adopting from this cell:

**1. Import everything up front.** Notebook 01 imported as it went, which is fine for a tutorial
but makes a notebook hard to re-use. One import cell at the top is cleaner.

**2. Seed the random number generators.** PyTorch initialises weights randomly and the
`DataLoader` shuffles randomly. `torch.manual_seed(42)` makes both reproducible, so your numbers
match your classmates' (and your own second run).

**3. Pick the device once.** The line

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

is the standard PyTorch idiom. Write your code against `device` and the *same* notebook runs on a
laptop CPU and on a GPU cluster with no edits. The rule to remember: **the model and its input
batch must be on the same device**, otherwise PyTorch raises
`RuntimeError: Expected all tensors to be on the same device`.

In [ ]:
# --- core PyTorch ---
import torch
import torch.nn as nn                    # layers: Linear, Dropout, ...
import torch.nn.functional as F          # stateless functions: relu, softmax, ...
import torch.optim as optim              # optimizers: Adam, SGD, ...
from torch.utils.data import DataLoader, random_split

# --- datasets and image transforms ---
import torchvision
from torchvision import datasets, transforms

# --- plotting and metrics ---
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Fix the random seeds so this notebook is REPRODUCIBLE: the same weight initialisation
# and the same shuffling order every time you run it.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Choose the hardware ONCE, then write all code against `device`.
# This exact line runs unchanged on a CPU-only laptop and on a GPU machine.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version :", torch.__version__)
print("torchvision     :", torchvision.__version__)
print("Using device    :", device)
if device.type == "cuda":
    print("GPU name        :", torch.cuda.get_device_name(0))
else:
    print("(No GPU found - that is fine, this small network trains quickly on the CPU.)")

### 1.2 Define the preprocessing transforms

A **transform** is a small function applied to every image as it is loaded. We chain two of them
with `transforms.Compose`, and the order matters:

**1. `transforms.ToTensor()`** does three things at once:

* converts the PIL image to a `torch.FloatTensor`,
* rescales the pixel values from integers `0…255` to floats `0.0…1.0` (a free division by 255),
* reorders the axes to PyTorch's image convention **`[channels, height, width]`** — so one MNIST
  image becomes shape `[1, 28, 28]`.

**2. `transforms.Normalize((mean,), (std,))`** applies the same z-score idea we used on the iris
features, but per **channel** instead of per column:

$$ x_{\text{norm}} = \frac{x - \mu}{\sigma} $$

The constants `0.1307` and `0.3081` are the well-known mean and standard deviation of the MNIST
training set. (Most images are mostly black background, which is why the mean is so far below
0.5.) After this step pixel values run roughly from `-0.42` to `+2.82`, centred near zero —
exactly the input range neural networks train best on.

> **Why normalise images at all?** The same reason as in Notebook 01: inputs centred on zero with
> unit spread give better-behaved gradients and faster convergence. It is not optional folklore —
> try commenting out `Normalize` in the exercises and watch the first epochs get worse.

> **Note on order:** `Normalize` expects a tensor, so `ToTensor()` **must** come first. Swapping
> the two lines raises a `TypeError`.

In [ ]:
# Well-known statistics of the MNIST training set (mean and std of the pixel values
# AFTER ToTensor has scaled them into the range [0, 1]). We store them in variables
# because we will need them again later to UN-normalise images for plotting.
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

# transforms.Compose chains transforms and applies them in the given order.
transform = transforms.Compose([
    # 1) PIL image (28x28, values 0..255) -> FloatTensor of shape [1, 28, 28], values 0.0..1.0
    transforms.ToTensor(),

    # 2) z-score per channel: x -> (x - mean) / std.  MNIST is grayscale, so the tuples
    #    have a single element. For an RGB dataset you would pass three values each.
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

print("Transform pipeline:")
print(transform)

### 1.3 Download the dataset

`datasets.MNIST` is a **`Dataset`** object — the first half of PyTorch's data pipeline. A
`Dataset` only has to answer two questions:

* `len(dataset)` → how many samples are there?
* `dataset[i]` → give me sample *i*, as a `(input, label)` tuple.

That is the whole interface. Every dataset you ever write yourself will implement just those two
methods.

The arguments:

| Argument | Meaning |
|---|---|
| `root='./data'` | folder to store the files in (created if missing) |
| `train=True` / `False` | pick the official 60 000-image training half or the 10 000-image test half |
| `download=True` | fetch the files if they are not already in `root` (needs internet the first time only) |
| `transform=transform` | apply our `ToTensor` + `Normalize` pipeline to every image on access |

> **Important:** the transform is applied **lazily**, i.e. when you index the dataset, not when you
> create it. This is what makes large datasets possible — nothing is preprocessed until it is
> actually needed.

The first run prints download progress bars; later runs find the cached files and start instantly.

In [ ]:
# Download (or load from cache) the official MNIST training half: 60,000 images.
train_full = datasets.MNIST(
    root='./data',          # where to store / look for the files
    train=True,             # the training half
    download=True,          # fetch it if it is not there yet
    transform=transform,    # our ToTensor + Normalize pipeline, applied on access
)

# The official test half: 10,000 images we will only touch once, at the very end.
test_dataset = datasets.MNIST(
    root='./data',
    train=False,            # the test half
    download=True,
    transform=transform,
)

print(f"Training images (before we carve out a validation set): {len(train_full)}")
print(f"Test images:                                            {len(test_dataset)}")
print(f"Class names / labels: {train_full.classes}")

### 1.4 Inspect one sample, and the class balance

Never train on data you have not looked at. Two checks here:

**Shape and range of a single image.** `train_full[0]` returns a `(image, label)` tuple. Expect:

* `image.shape` → `torch.Size([1, 28, 28])` — read as *1 channel, 28 rows, 28 columns*. Note there
  is **no batch dimension** yet; the `DataLoader` adds that in Step 2.
* `image.dtype` → `torch.float32`, and values roughly in `[-0.42, +2.82]` because of `Normalize`.
  If you see `0…255` or `0…1`, your transform did not run.
* `label` → a plain Python `int`, e.g. `5`. The `DataLoader` will collate these into a `long`
  tensor for us, which is exactly what `CrossEntropyLoss` wants.

**Class balance.** MNIST is *near*-balanced: every digit has between about 5 400 and 6 700
training images. That matters for two reasons: plain **accuracy is a fair metric** here (unlike on
a heavily skewed dataset), and the baseline to beat is low: **10%** for a uniform random guess,
or **11.2%** if you always answer with the most common digit (`1`, with 6 742 training images).

In [ ]:
# --- one sample -----------------------------------------------------------------------
image, label = train_full[0]     # Dataset indexing returns an (input, label) tuple

print("--- A single MNIST sample ---")
print("image type :", type(image))
print("image shape:", image.shape)          # expect [1, 28, 28] = [channels, height, width]
print("image dtype:", image.dtype)          # expect torch.float32
print(f"pixel range: {image.min():.3f} .. {image.max():.3f}   (normalised, so negatives are normal)")
print("label      :", label, "-> type:", type(label).__name__)

# --- class balance --------------------------------------------------------------------
# .targets holds all 60,000 labels as a tensor, so we can count them without a loop.
counts = torch.bincount(train_full.targets)
print("\n--- How many training images per digit? ---")
for digit, n in enumerate(counts.tolist()):
    print(f"  digit {digit}: {n:5d} images  ({100 * n / len(train_full):.2f}%)")
print(f"\nBaseline to beat: always guessing one class gives ~{100 * counts.max().item() / len(train_full):.1f}% accuracy.")

### 1.5 Look at the digits

This is the most important sanity check of the whole notebook: **plot the data**. It catches
mislabelled datasets, broken transforms and wrong axis orders in seconds — bugs that otherwise
hide until your accuracy is mysteriously stuck.

Two mechanics to notice in the code:

* **Un-normalising for display.** Our images have been z-scored, so plotting them raw gives odd
  contrast and matplotlib warnings. We invert the transform with
  `image * std + mean` to get back to the `[0, 1]` range that `imshow` expects.
* **Dropping the channel dimension.** `imshow` wants a 2-D array `[height, width]`, but our tensor
  is `[1, 28, 28]`. `.squeeze()` removes the size-1 channel axis.

Look at the digits and appreciate how *varied* human handwriting is — some 4s look like 9s, some
1s are drawn with a base serif and some without. That variability is the actual difficulty of the
task.

In [ ]:
def show_digits(dataset, indices, titles=None, ncols=8, suptitle=None):
    '''Plot MNIST images from a dataset on a grid, un-normalising them first.'''
    nrows = int(np.ceil(len(indices) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.4 * ncols, 1.6 * nrows))
    axes = np.array(axes).reshape(-1)          # flatten the axes grid to a 1-D list

    for ax_i, ax in enumerate(axes):
        ax.axis('off')                         # hide the pixel ruler on every panel
        if ax_i >= len(indices):
            continue                           # leave the leftover panels blank
        img, lab = dataset[indices[ax_i]]

        # Undo Normalize so the image is back in [0, 1], then drop the channel axis:
        # [1, 28, 28] -> [28, 28], which is what imshow expects for a grayscale image.
        img = img * MNIST_STD + MNIST_MEAN
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(titles[ax_i] if titles is not None else f"label: {lab}", fontsize=9)

    if suptitle:
        fig.suptitle(suptitle, fontsize=14)
    plt.tight_layout()
    plt.show()


# Show the first 16 training images with their labels. Check that every label really
# matches the digit you see - if not, something is wrong with the data pipeline.
show_digits(train_full, indices=range(16),
            suptitle='MNIST training samples (with their true labels)')

### 1.6 The key idea: flattening an image for a feed-forward network

An `nn.Linear` layer accepts a **flat vector**, not a 2-D picture. So before the first layer we
reshape every image:

```
[batch, 1, 28, 28]   ──flatten──▶   [batch, 784]          because 1 x 28 x 28 = 784
```

The pixels are simply read out row by row, top-left to bottom-right, and lined up in a row.

### What this costs us — read this carefully

Flattening throws away **all spatial structure**. After the flatten, the network has no idea that
pixel 0 and pixel 1 were neighbours while pixel 0 and pixel 28 were vertically adjacent. To the
FNN, an image is just 784 unrelated numbers in a fixed order. Two consequences:

1. **No translation invariance.** Shift the same digit two pixels to the right and *every* input
   number changes position. The network has to learn "a 7 up here" and "a 7 slightly lower" as two
   separate patterns, from separate examples.
2. **Lots of parameters.** Our first layer alone needs 784 × 256 ≈ 201 000 weights, because every
   pixel is wired to every hidden neuron.

It still works well on MNIST — the digits are pre-centred and size-normalised, which quietly hides
problem #1. On real-world photographs an FNN falls apart, and that is precisely the gap that
**convolutional layers** (later in the course) are designed to fill. For now, notice the
limitation and keep it in mind.

### Where to put the flatten?

Both of these are correct and common:

* **inside `forward()`** with `x.view(x.size(0), -1)` or `torch.flatten(x, 1)` — what we do below;
* as a **layer**, `nn.Flatten()`, which is handy inside `nn.Sequential`.

The `1` in `torch.flatten(x, 1)` means *"start flattening at dimension 1"* — i.e. leave dimension
0, the batch dimension, untouched. **Never flatten the batch dimension**; that would merge all your
samples into one giant vector. This is one of the most common shape bugs in PyTorch.

In [ ]:
# A concrete demonstration of the flatten, on a small fake batch of 4 images.
demo_batch = torch.stack([train_full[i][0] for i in range(4)])   # stack adds a batch dim

print("Before flatten:", demo_batch.shape, " = [batch, channels, height, width]")

# torch.flatten(x, 1) collapses every dimension FROM index 1 onwards into one.
# Dimension 0 (the batch) is deliberately left alone.
flat = torch.flatten(demo_batch, 1)
print("After  flatten:", flat.shape, "      = [batch, 1*28*28] = [batch, 784]")

print("\n28 * 28 =", 28 * 28, "-> this is why the first Linear layer must have in_features=784")

# WRONG version, shown so you recognise the bug when you make it:
# it merges all 4 samples into a single vector and the model would then see one "sample".
print("\nWrong (flattens the batch away too):", torch.flatten(demo_batch).shape)

---
# Step 2 · Split the Data and Build DataLoaders

**Goal:** create the three splits, then wrap them in `DataLoader`s that serve shuffled
mini-batches.

### 2.0 Which splits do we need?

| Split | Size | Where it comes from |
|---|---|---|
| **Training** | 55 000 | the official 60 000, minus the validation slice |
| **Validation** | 5 000 | carved out of the official training half with `random_split` |
| **Test** | 10 000 | the **official** MNIST test set, untouched until Step 6 |

Same logic as Notebook 01 — train on one, tune on another, report on a third — but note the
scale: 5 000 validation images give a resolution of 0.02% per image, instead of the jumpy 6.7% we
had with 15 iris flowers. The metrics in this notebook are **far** more trustworthy.

`random_split(dataset, [55000, 5000])` returns two `Subset` objects that reference the original
dataset without copying any image data. The sizes must sum exactly to `len(dataset)` or it raises
an error. We pass a `generator` seeded with 42 so the split is reproducible.

> **Note:** unlike `train_test_split`, `random_split` has no `stratify` option. With 5 000 random
> samples out of a near-balanced 60 000, the class proportions come out close to even anyway — but
> on a small or imbalanced dataset you would need `StratifiedShuffleSplit` instead.

In [ ]:
# Decide the sizes first, computing the training size so the two always sum to 60,000.
VAL_SIZE = 5_000
TRAIN_SIZE = len(train_full) - VAL_SIZE      # 60,000 - 5,000 = 55,000

# random_split shuffles the indices and hands back two Subset views of `train_full`.
# The generator makes the split reproducible - without it you would get a different
# 55k/5k division on every run, and your validation curve would not be comparable.
train_dataset, val_dataset = random_split(
    train_full,
    [TRAIN_SIZE, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Train      : {len(train_dataset):6d} images")
print(f"Validation : {len(val_dataset):6d} images")
print(f"Test       : {len(test_dataset):6d} images   (official MNIST test set)")
print(f"Total      : {len(train_dataset) + len(val_dataset) + len(test_dataset):6d}")

# Confirm the validation slice is not accidentally lopsided towards some digits.
val_labels = torch.tensor([train_full.targets[i].item() for i in val_dataset.indices])
print("\nValidation label counts per digit:", torch.bincount(val_labels).tolist())

### 2.1 DataLoaders — the second half of the data pipeline

A `Dataset` gives you **one sample at a time**. A **`DataLoader`** wraps it and gives you
**batches**, handling four jobs for you:

| Job | Controlled by |
|---|---|
| Group samples into batches and stack them into one tensor | `batch_size` |
| Shuffle the order every epoch | `shuffle=True` |
| Load in background worker processes so the GPU never waits | `num_workers` |
| Iterate exactly once over the whole split | `for batch in loader:` |

### Why mini-batches? (the big fix from Notebook 01)

In Notebook 01 we pushed all 120 iris samples through at once, so one epoch = **one** weight
update, and 100 epochs left the model underfitted. Here:

$$ \text{updates per epoch} = \left\lceil \frac{55\,000}{64} \right\rceil = 860 $$

so **15 epochs give 12 900 weight updates** instead of 100. That is the single biggest reason this
notebook converges properly while Notebook 01 did not.

Mini-batches are a compromise between two extremes:

| | Updates per epoch | Gradient quality | Speed |
|---|---|---|---|
| **Full batch** (all 55 000) | 1 | exact | slow to converge; may not fit in memory |
| **Mini-batch** (64) | 860 | noisy estimate | ✅ the practical sweet spot |
| **Stochastic / single sample** | 55 000 | very noisy | poor hardware utilisation |

The gradient from 64 samples is only an *estimate* of the true gradient — and that noise is
actually useful: it helps the optimizer escape poor local minima.

### Why `shuffle=True` on the training set only?

Shuffling breaks any ordering in the data, so each batch is a representative mixture of digits and
consecutive updates are not correlated. For validation and test we only *measure*, and the mean
does not depend on the order — so `shuffle=False` keeps our predictions aligned with the dataset
indices, which we will rely on later when we plot the misclassified digits.

> **Colab tip:** `num_workers=2` uses two background processes to prepare batches. If you ever get
> worker crashes or hangs (common on Windows and inside some notebook environments), just set
> `num_workers=0`.

In [ ]:
BATCH_SIZE = 64      # how many images the network sees per weight update

# --- training loader: shuffled, because the ORDER of updates matters while learning ---
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,        # reshuffle every epoch -> each batch is a fresh mix of digits
    num_workers=2,       # background processes preparing batches (set 0 if you hit problems)
)

# --- validation / test loaders: no shuffling needed, we are only measuring ------------
# A larger batch size is fine (and faster) here because no gradients are stored.
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print(f"Batches per training epoch: {len(train_loader)}   "
      f"(= ceil({TRAIN_SIZE} / {BATCH_SIZE}))")
print(f"Batches per validation pass: {len(val_loader)}")
print(f"Batches per test pass:       {len(test_loader)}")

# --- inspect ONE batch ----------------------------------------------------------------
# next(iter(loader)) is the standard trick to peek at a single batch without a full loop.
images, labels = next(iter(train_loader))
print("\n--- One training batch ---")
print("images shape:", images.shape, "= [batch, channels, height, width]")
print("labels shape:", labels.shape, "= one label per image")
print("labels dtype:", labels.dtype, "-> torch.int64, exactly what CrossEntropyLoss needs")
print("labels in this batch:", labels[:16].tolist(), "... (mixed digits, thanks to shuffle=True)")

# The last batch of an epoch is usually SMALLER, because 55,000 is not divisible by 64.
print(f"\n{TRAIN_SIZE} / {BATCH_SIZE} = {TRAIN_SIZE / BATCH_SIZE:.2f}"
      f"  -> the final batch holds only {TRAIN_SIZE % BATCH_SIZE} images.")
print("Remember this when averaging losses: weight each batch by its actual size!")

---
# Step 3 · Define the Feed-Forward Network

**Goal:** describe the architecture. Structurally this is the *same* class you wrote in Notebook
01 — `nn.Module`, layers in `__init__`, data path in `forward` — with three additions: a flatten,
wider layers, and Dropout.

### Our architecture

| Layer | Definition | Shape transformation | Parameters |
|---|---|---|---|
| flatten | `torch.flatten(x, 1)` | `[B, 1, 28, 28]` → `[B, 784]` | 0 |
| `fc1` | `nn.Linear(784, 256)` | `[B, 784]` → `[B, 256]` | 784×256 + 256 = **200 960** |
| ReLU + Dropout(0.2) | | unchanged | 0 |
| `fc2` | `nn.Linear(256, 128)` | `[B, 256]` → `[B, 128]` | 256×128 + 128 = **32 896** |
| ReLU + Dropout(0.2) | | unchanged | 0 |
| `fc3` | `nn.Linear(128, 10)` | `[B, 128]` → `[B, 10]` | 128×10 + 10 = **1 290** |

**Total: 235 146 learnable parameters** — about 1 200× more than `IrisNet`, and yet still small
enough to train on a CPU in minutes.

Notice the **funnel shape** 784 → 256 → 128 → 10. Each layer compresses the representation a bit
more, forcing the network to keep only what is useful for telling digits apart. This is a common
design heuristic, not a law — the exercises invite you to try other widths.

### Why the output layer has 10 neurons

One per class, and — exactly as in Notebook 01 — **no softmax**, because `nn.CrossEntropyLoss`
applies log-softmax internally. The 10 raw outputs are *logits*; the largest one is the prediction.

### New: Dropout

`nn.Dropout(p=0.2)` randomly zeroes 20% of the values passing through it, choosing different ones
every forward pass, and scales the survivors up by `1/(1-p)` to keep the average magnitude
constant.

**Why on earth would we damage our own signal?** Because it prevents the network from relying on
any single neuron. Each neuron must contribute something useful on its own, which produces a more
robust, less over-fitted model. Think of it as training a huge ensemble of slightly different
sub-networks that share weights.

**The crucial detail:** Dropout must be **active while training** and **switched off when
evaluating** — you want all your neurons working when making a real prediction. PyTorch handles
this through the mode flag:

| Call | Dropout behaviour |
|---|---|
| `model.train()` | ON — randomly zeroes activations |
| `model.eval()` | OFF — passes everything through, unchanged |

In Notebook 01 those two calls were pure habit-building; **now they change the numbers**. Forget
`model.eval()` before evaluating and your accuracy will be randomly worse on every run — a
genuinely nasty bug to track down.

### 3.1 Write the model class

Read it in the same order as before: `super().__init__()` first, then the layers, then the data
path in `forward`. The one new line to trace carefully is the flatten at the top of `forward` —
that is the bridge between "image" and "vector".

Note the layer widths chain correctly: `784 → 256`, `256 → 128`, `128 → 10`. If you change one
number you must change its neighbour, or PyTorch will raise a shape error on the first forward
pass.

In [ ]:
class MnistFNN(nn.Module):
    '''A fully-connected (feed-forward) classifier for MNIST: 784 -> 256 -> 128 -> 10.

    No convolutions anywhere - every pixel is treated as an independent input feature.
    '''

    def __init__(self, dropout_p=0.2):
        super(MnistFNN, self).__init__()          # ALWAYS first: sets up nn.Module

        # The three fully-connected layers. 784 = 1 * 28 * 28 pixels per image.
        self.fc1 = nn.Linear(28 * 28, 256)        # 200,960 parameters
        self.fc2 = nn.Linear(256, 128)            #  32,896 parameters
        self.fc3 = nn.Linear(128, 10)             #   1,290 parameters -> one logit per digit

        # Dropout is a LAYER with state (its on/off mode), so it belongs in __init__.
        # We reuse the same module twice in forward(); that is fine because it holds
        # no learnable weights.
        self.dropout = nn.Dropout(p=dropout_p)

    def forward(self, x):
        '''x arrives with shape [batch, 1, 28, 28] and leaves as [batch, 10] logits.'''

        # STEP 1: flatten each image into a vector so nn.Linear can accept it.
        # The `1` means "start at dimension 1", which LEAVES THE BATCH DIMENSION ALONE.
        x = torch.flatten(x, 1)                   # [B, 1, 28, 28] -> [B, 784]

        # STEP 2: hidden layer 1 -> ReLU -> Dropout
        x = F.relu(self.fc1(x))                   # [B, 784] -> [B, 256]
        x = self.dropout(x)                       # active in train() mode, off in eval() mode

        # STEP 3: hidden layer 2 -> ReLU -> Dropout
        x = F.relu(self.fc2(x))                   # [B, 256] -> [B, 128]
        x = self.dropout(x)

        # STEP 4: output layer. NO activation and NO softmax on purpose -
        # these 10 raw logits go straight into nn.CrossEntropyLoss.
        x = self.fc3(x)                           # [B, 128] -> [B, 10]
        return x


print("MnistFNN class defined.")

### 3.2 Instantiate the model and move it to the device

Two things happen in the cell below that did not happen in Notebook 01:

**`model.to(device)`** moves every parameter tensor to the GPU (if there is one). Remember the
rule: **model and data on the same device.** We will call `.to(device)` on each batch inside the
training loop for exactly that reason. (For a model, `.to()` also works in place, so
`model.to(device)` alone is enough — but assigning the result is the clearer habit, and it is
required for plain tensors.)

**The parameter count** is worth computing yourself rather than trusting the table above:

```python
sum(p.numel() for p in model.parameters() if p.requires_grad)
```

`p.numel()` is the number of elements in a tensor, and `requires_grad` filters to the *learnable*
ones. Expect **235 146**.

Finally, we run a single dummy batch through the untrained model. This **shape test** takes one
second and catches architecture mistakes before you waste minutes of training on them:

* output shape must be `[batch, 10]`;
* accuracy right now should be around **10%** — random guessing over 10 classes.

> **Remember from Notebook 01:** re-running this cell resets the weights, so re-run the optimizer
> cell (Step 4) too — the optimizer holds a reference to *this* model's parameters.

In [ ]:
# Create the network and move all 235,146 parameters onto the chosen device.
model = MnistFNN(dropout_p=0.2).to(device)

# Print the layer-by-layer structure. Reading this output is a good way to double-check
# that the widths chain as you intended.
print(model)

# Count the learnable parameters. p.numel() = number of elements in tensor p.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")
print("  fc1:", 28 * 28 * 256 + 256)
print("  fc2:", 256 * 128 + 128)
print("  fc3:", 128 * 10 + 10)

# --- SHAPE TEST: always run one dummy batch through a fresh model -----------------------
# This costs a second and catches architecture bugs before you spend minutes training.
model.eval()                                  # eval mode -> Dropout off, so this is deterministic
with torch.no_grad():
    dummy = images[:8].to(device)             # 8 images from the batch we peeked at earlier
    logits = model(dummy)

print("\n--- Shape test on an untrained model ---")
print("input :", tuple(dummy.shape), "-> output:", tuple(logits.shape), "(expect [8, 10])")
print("logits for the first image:", logits[0].cpu().numpy().round(3))
print("predicted digit:", logits[0].argmax().item(), "| true digit:", labels[0].item())
print("(The prediction is meaningless: the weights are still random -> expect ~10% accuracy.)")

---
# Step 4 · Loss Function and Optimizer

**Goal:** define *how wrong* the model is and *how* it should improve. This step is **identical**
to Notebook 01 — only the number of classes changed. That is the point of learning a recipe.

### `nn.CrossEntropyLoss`

Still the right choice: multi-class, single-label classification. It consumes

* **logits** of shape `[batch, 10]` — raw, no softmax,
* **labels** of shape `[batch]` — integer class indices, dtype `long`.

**Your sanity-check number:** a model guessing uniformly over 10 classes has a loss of

$$ -\ln\!\left(\tfrac{1}{10}\right) = \ln 10 \approx 2.303 $$

So the *first* loss you see should be near 2.30. If epoch 1 prints something wildly larger (say
15), something is broken — usually unnormalised inputs or a much too large learning rate. Watch
for this; it is the fastest bug detector you have.

(For comparison, Notebook 01's 3-class baseline was $-\ln(1/3) \approx 1.10$. More classes ⇒
higher starting loss.)

### `optim.Adam(lr=0.001)`

Unchanged from Notebook 01, and `0.001` is Adam's default for good reason — it is a solid starting
point for most problems. With 860 updates per epoch this learning rate is now *plenty*; you will
see the loss drop below 0.4 within the first epoch, unlike the crawl we suffered last time.

In [ ]:
# Identical to Notebook 01 - the recipe does not change with the dataset.
criterion = nn.CrossEntropyLoss()

LEARNING_RATE = 0.001
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Loss     :", criterion)
print("Optimizer:", optimizer.__class__.__name__, f"(lr={LEARNING_RATE})")

# The sanity-check baseline: 10 classes -> a uniform guesser scores -ln(1/10) = 2.303.
print(f"\nExpected loss BEFORE any training: {np.log(10):.4f}")

# Let us verify that on the actual untrained model, using one batch.
model.eval()
with torch.no_grad():
    batch_loss = criterion(model(images.to(device)), labels.to(device))
print(f"Measured loss on one batch:        {batch_loss.item():.4f}  <- should be close to 2.30")

---
# Step 5 · Train the Network

**Goal:** run mini-batch gradient descent and record the history so we can plot it.

### The loop now has two levels

```
for epoch in range(EPOCHS):              # outer: 15 passes over the data
    for images, labels in train_loader:  # inner: 860 mini-batches per pass
        zero_grad -> forward -> loss -> backward -> step     # ONE weight update
    evaluate on the validation set                            # once per epoch
```

The four lines you memorised in Notebook 01 are **unchanged** — they have simply moved into the
inner loop, so they now run 12 900 times instead of 100.

### Three new details that matter

**1. Move each batch to the device.**

```python
images, labels = images.to(device), labels.to(device)
```

The `DataLoader` always produces CPU tensors. If your model is on the GPU and you forget this
line, you get `RuntimeError: Expected all tensors to be on the same device`.

**2. Average the epoch loss correctly.** Each batch gives a loss that is already the *mean* over
its own samples. Averaging those means directly is subtly wrong, because the last batch holds only
24 images and would count as much as a full batch of 64. So we accumulate a **weighted** sum:

```python
running_loss += loss.item() * labels.size(0)   # mean x batch size = sum over the batch
...
epoch_loss = running_loss / total_samples      # divide by the true sample count
```

With 860 batches the difference is small, but the habit is correct and it matters on small
datasets.

**3. `loss.item()`, not `loss`.** Appending the *tensor* would keep its whole computation graph
alive and leak memory across the epoch. `.item()` extracts a plain Python float, dropping the
graph.

### About the training accuracy we print

It is measured **with Dropout active** (we are in `train()` mode), so it slightly *understates*
the model's real ability. Do not be surprised when training accuracy sits a little *below*
validation accuracy in the early epochs — that is Dropout, not a bug. It is also the reason the
two curves will look unusually close together in Step 7.

### What you should see

| | Expected |
|---|---|
| Epoch 1 | train loss drops from ~2.3 to roughly **0.29**; validation accuracy already **~95%** |
| Epoch 15 | train loss around **0.035–0.05**, validation accuracy **~97.9%** |
| Best epoch | validation accuracy peaks near **0.979**, often *before* the last epoch |
| Time | a few seconds per epoch on a GPU, ~15–25 s per epoch on a Colab CPU |

Compare that with Notebook 01, where after 100 epochs the loss had barely moved from its baseline.
Same recipe, same optimizer, same learning rate — **the difference is mini-batching**.

### 5.1 An `evaluate()` helper function

We need the exact same "measure loss and accuracy over a whole split" logic three times: for
validation each epoch, and for the test set at the end. So we write it **once** as a function.

Extracting it is not just tidiness — it guarantees validation and test are measured *identically*,
which is the kind of consistency that makes results trustworthy.

Everything that makes this a measurement rather than a training step is inside:

* `model.eval()` → Dropout off,
* `torch.no_grad()` → no computation graph, less memory, faster,
* no `backward()`, no `step()` → nothing is learned.

In [ ]:
def evaluate(model, loader, criterion, device):
    '''Run the model over a whole DataLoader and return (average_loss, accuracy).

    No gradients, no weight updates - this only MEASURES.
    '''
    model.eval()                 # Dropout OFF: we want all neurons for a real prediction
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():        # no computation graph -> faster and lighter
        for images, labels in loader:
            # Batches come from the DataLoader on the CPU; move them where the model is.
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)                  # [batch, 10] logits
            loss = criterion(outputs, labels)

            # loss is the MEAN over this batch, so multiply by the batch size to get the
            # sum. This keeps the average correct even though the last batch is smaller.
            running_loss += loss.item() * labels.size(0)

            # Prediction = index of the largest logit, taken along the class dimension.
            predicted = outputs.argmax(dim=1)        # same as torch.max(outputs, 1)[1]
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


# Quick check on the untrained model: expect loss ~2.30 and accuracy ~10% (1 in 10 classes).
untrained_loss, untrained_acc = evaluate(model, val_loader, criterion, device)
print(f"Untrained model -> val loss: {untrained_loss:.4f}, val accuracy: {untrained_acc:.4f}")
print("(Accuracy near 0.10 confirms the model is still guessing at random.)")

### 5.2 The training loop

This is the cell that does the work. Trace the five numbered steps in the inner loop and compare
them line by line with Notebook 01 — they are the same five steps.

In [ ]:
EPOCHS = 15

# One history dictionary is tidier than four separate lists.
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f"Training for {EPOCHS} epochs on {device}, "
      f"{len(train_loader)} mini-batches per epoch "
      f"({EPOCHS * len(train_loader):,} weight updates in total).\n")

for epoch in range(1, EPOCHS + 1):

    # ==================== TRAINING PHASE (one pass over the 55,000 images) =============
    model.train()                       # Dropout ON - this now genuinely changes behaviour
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        # Move this mini-batch to the same device as the model.
        images, labels = images.to(device), labels.to(device)

        # STEP 1: clear the gradients from the PREVIOUS batch (PyTorch accumulates them).
        optimizer.zero_grad()

        # STEP 2: forward pass -> [64, 10] logits.
        outputs = model(images)

        # STEP 3: how wrong is this batch?
        loss = criterion(outputs, labels)

        # STEP 4: backpropagate -> fills p.grad for all 235,146 parameters.
        loss.backward()

        # STEP 5: update the weights. This runs 860 times per epoch, not once!
        optimizer.step()

        # --- bookkeeping (no learning happens here) ---
        # Weight the batch mean by the batch size so the epoch average is exact.
        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total          # slightly pessimistic: measured with Dropout ON

    # ==================== VALIDATION PHASE (once per epoch) ===========================
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    # Record everything for the plots in Step 7.
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch [{epoch:2d}/{EPOCHS}]  "
          f"train loss: {train_loss:.4f}  train acc: {train_acc:.4f}  |  "
          f"val loss: {val_loss:.4f}  val acc: {val_acc:.4f}")

print("\nTraining complete.")
print(f"Best validation accuracy: {max(history['val_acc']):.4f} "
      f"at epoch {int(np.argmax(history['val_acc'])) + 1}")

---
# Step 6 · Evaluate on the Official Test Set

**Goal:** one honest number, from the 10 000 images that influenced neither the weights nor a
single one of our design decisions.

We reuse the `evaluate()` helper — the *same* code path as validation, which is precisely why we
wrote it as a function.

### What to expect, and how to read it

A well-trained FNN of this size scores about **97.5–98.5%** on the MNIST test set (a verified run
of this exact notebook reached **98.06%**, i.e. 194 mistakes out of 10 000). Two useful reference
points to put that in context:

| Model | Typical test accuracy |
|---|---|
| Always guess one digit | ~11% |
| Logistic regression (no hidden layer) | ~92% |
| **This FNN (784-256-128-10)** | **~98%** |
| A small CNN (later in the course) | ~99.2% |
| Best published results | ~99.8% |

The gap between our ~98% and a CNN's ~99.2% may look tiny, but read it as an **error rate**: 2%
versus 0.8% means the CNN makes *less than half* as many mistakes. That is the payoff for
respecting the spatial structure our flatten threw away.

Also note how much closer test and validation accuracy are here than in Notebook 01. With 10 000
test images instead of 15, the estimate is stable to within a few tenths of a percent — small
splits, not overfitting, were the real problem last time.

### Beyond overall accuracy: per-class accuracy

One aggregate number hides *where* the errors are. Computing accuracy **per digit** tells you
which classes are hard. On MNIST you will typically find **0** and **1** highest (~99.2–99.3%,
they have distinctive shapes) while **5** and **8** trail by two full percentage points (~96.5%) —
they are the digits with the most ambiguous handwritten forms. The spread between the best and
worst digit is larger than the gap between this model and a CNN, which is worth remembering before
you chase the last fraction of overall accuracy.

### 6.1 Collect predictions on the test set

We run the test set once and store **everything** we might want to analyse: the predicted labels,
the true labels, and the full softmax probabilities.

This is where `shuffle=False` pays off. Because the loader preserved the dataset order,
`all_preds[i]` corresponds to `test_dataset[i]` — which lets us look up and plot the exact images
the model got wrong in Step 7.3.

Note `torch.softmax(outputs, dim=1)`: this is the one place we *do* want an explicit softmax,
because we want interpretable **probabilities** (each row sums to 1) rather than raw logits. It
never goes near the loss function.

In [ ]:
# The headline number, via the same helper we used for validation each epoch.
test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}  ({test_accuracy * 100:.2f}%)")
print(f"-> the model gets {round((1 - test_accuracy) * len(test_dataset))} "
      f"of {len(test_dataset)} test images wrong.\n")

# --- collect the detailed predictions for the analysis and plots below ----------------
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images_b, labels_b in test_loader:      # shuffle=False -> order matches test_dataset
        outputs = model(images_b.to(device))

        # softmax turns the 10 logits into 10 probabilities that sum to 1.
        # We only do this for INTERPRETATION - never before CrossEntropyLoss.
        probs = torch.softmax(outputs, dim=1)

        # Bring results back to the CPU so we can concatenate and use NumPy/sklearn.
        all_probs.append(probs.cpu())
        all_preds.append(outputs.argmax(dim=1).cpu())
        all_labels.append(labels_b)

# torch.cat glues the per-batch pieces into one long tensor each.
all_preds = torch.cat(all_preds)      # [10000]
all_labels = torch.cat(all_labels)    # [10000]
all_probs = torch.cat(all_probs)      # [10000, 10]

print("Collected predictions:", all_preds.shape, "| probabilities:", all_probs.shape)
# Cross-check: this must equal the accuracy printed above.
print("Accuracy recomputed from the stored predictions:",
      f"{(all_preds == all_labels).float().mean().item():.4f}")

### 6.2 Per-class accuracy — where are the mistakes?

For each digit *d* we count how many test images with true label *d* were predicted correctly.
Read the output as a difficulty ranking of the ten digits, and keep it in mind when you look at
the confusion matrix next — the low scorers here will be the rows with the most off-diagonal
weight there.

In [ ]:
print("Per-class accuracy on the test set:\n")
print("digit   correct / total   accuracy")
print("-" * 38)

for digit in range(10):
    mask = (all_labels == digit)                       # boolean mask: which rows are this digit
    n_total = mask.sum().item()
    n_correct = (all_preds[mask] == digit).sum().item()
    acc = n_correct / n_total

    # A crude text bar makes the ranking easy to see at a glance.
    bar = "#" * int(round(acc * 40))
    print(f"  {digit}     {n_correct:4d} / {n_total:4d}      {acc:.4f}  {bar}")

print("-" * 38)
print(f"overall {(all_preds == all_labels).sum().item():5d} / {len(all_labels)}"
      f"      {test_accuracy:.4f}")
print("\nTypically 0 and 1 score highest (distinctive shapes), while 8, 9 and 5 are hardest.")

---
# Step 7 · Visualise the Training History and the Errors

**Goal:** three views that a single accuracy number cannot give you — *how* training went, *which*
digits get confused, and *what* the failures actually look like.

### 7.1 Loss and accuracy curves

Same two panels as Notebook 01, and the same diagnostic table applies:

| What you see | Diagnosis |
|---|---|
| Both losses still falling steeply at the end | **Underfitting** — train longer (this was Notebook 01) |
| Training loss ↓ while validation loss ↑ | **Overfitting** — the model is memorising |
| Both low, small gap | 🎉 Well fitted |

**What to look for this time.** The loss drops off a cliff in the first epoch or two and then
flattens — that is healthy convergence, and a completely different picture from Notebook 01.

Now look at the **two loss curves separately after about epoch 4**. The training loss keeps
falling steadily (roughly 0.07 → 0.04), while the validation loss **stops improving and just
bounces around 0.08**. Neither curve is dramatic, but the *gap between them is widening* — and
that widening gap is the signature of **overfitting** beginning: with 235 146 parameters and
55 000 images, the network has enough capacity to start memorising training images rather than
learning general rules.

> **Do not expect a textbook picture here.** Diagrams in books show the validation loss turning
> cleanly upward. In reality it is **noisy**: it will jump up and down from epoch to epoch, and
> the lowest value may well land on the very last epoch by luck. Judge the trend, never a single
> epoch. Dropout is also actively holding the overfitting back — Exercise 7 removes it, and then
> you *will* see the classic upward turn.

The standard way to exploit this is **early stopping**: keep the weights from the epoch with the
best validation loss instead of whatever the last epoch happened to give you. That is Exercise 5,
and note from the training printout that the best epoch is usually *not* the final one.

One curiosity in the accuracy panel: **training accuracy starts *below* validation accuracy**
(epoch 1 is roughly 0.91 vs 0.95) and only overtakes it around epoch 5. That is not a bug.
Training accuracy is measured with Dropout **active** (a handicapped network), validation with
Dropout **off** (the full network). Once the model starts to overfit, the handicap is no longer
enough to hide it and the training curve pulls ahead.

In [ ]:
epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(13, 5))

# --------- Panel 1: loss -------------------------------------------------------------
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], marker='o', ms=3, label='Training Loss')
plt.plot(epochs_range, history['val_loss'], marker='s', ms=3, label='Validation Loss')

# Draw the random-guessing baseline: a uniform guesser over 10 classes scores ln(10).
plt.axhline(np.log(10), color='gray', ls='--', lw=1, label='Random guessing (ln 10 = 2.303)')

plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(alpha=0.3)
# Look for the point where the validation curve turns UP while training keeps falling:
# that is where overfitting starts, and where early stopping would cut training short.

# --------- Panel 2: accuracy ---------------------------------------------------------
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_acc'], marker='o', ms=3, label='Training Accuracy')
plt.plot(epochs_range, history['val_acc'], marker='s', ms=3, label='Validation Accuracy')

# Mark the final test accuracy so all three splits are comparable in one glance.
plt.axhline(test_accuracy, color='green', ls=':', lw=1.5,
            label=f'Test accuracy ({test_accuracy:.4f})')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
# Training accuracy below validation accuracy is EXPECTED here: training is measured
# with Dropout ON (handicapped), validation with Dropout OFF (full network).

plt.suptitle('MNIST Feed-Forward Network: Training History', fontsize=15)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### 7.2 The 10 × 10 confusion matrix

With ten classes the confusion matrix becomes genuinely informative: 100 cells, of which only 10
should hold large numbers.

**How to read it:** row = **true** digit, column = **predicted** digit. The diagonal is correct;
every off-diagonal cell names a specific confusion. Cell `(4, 9)` holding the value 12 means
*"twelve real 4s were called 9s"*.

We normalise nothing here, but the diagonal dominates so strongly (≈98% of 10 000 images) that a
plain count plot makes the off-diagonal cells nearly invisible. Two remedies are used below:

* `normalize='true'` in a second panel shows **row-wise fractions** — i.e. *recall* per digit —
  which makes small error counts visible.
* We print the largest off-diagonal entries as a plain ranked list, which is often easier to read
  than any heat map.

**Confusions to expect on MNIST.** The exact counts shift a little from run to run, but the
*pairs* are remarkably stable, and every one of them is a genuine shape similarity in handwriting:

* **5 → 3** — usually the biggest single entry: a 5 with a rounded upper stroke is a 3;
* **4 → 9** — close the top of a 4 and it becomes a 9;
* **7 ↔ 2** — depends entirely on whether the writer crossbars the 7 and curls the 2;
* **8 → 3** and **8 → 5** — an 8 whose left side is not fully closed;
* **9 → 3**, **3 → 2** — sloppy loops and curves.

The reassuring part: the model's mistakes are the mistakes a *human* would make on sloppy
handwriting, not random noise. That is evidence it learned something real about digit shape.

In [ ]:
# sklearn wants NumPy arrays, so convert the collected tensors.
y_true = all_labels.numpy()
y_pred = all_preds.numpy()

# cm[i, j] = how many images with TRUE label i were PREDICTED as j.
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# --------- Panel 1: raw counts -------------------------------------------------------
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10))).plot(
    ax=axes[0], cmap='Blues', colorbar=False, values_format='d'
)
axes[0].set_title('Confusion matrix - raw counts')

# --------- Panel 2: row-normalised (= recall per digit) -----------------------------
# normalize='true' divides each row by its total, so each row sums to 1.0. This makes
# the small error counts visible, which the huge diagonal hides in the left panel.
cm_norm = confusion_matrix(y_true, y_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=list(range(10))).plot(
    ax=axes[1], cmap='Blues', colorbar=False, values_format='.2f'
)
axes[1].set_title('Confusion matrix - normalised per true class (recall)')

plt.suptitle('MNIST Test Set: Which Digits Get Confused?', fontsize=15)
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()

# --------- The same information as a ranked list, which is often easier to read ------
print("Most frequent confusions (true digit -> predicted digit):\n")
mistakes = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j]
for count, true_d, pred_d in sorted(mistakes, reverse=True)[:10]:
    print(f"  {count:3d} times: a real {true_d} was classified as a {pred_d}")

### 7.3 Look at the mistakes

The single most useful debugging habit in deep learning: **look at what your model got wrong.**

Below we pick the errors the model was most *confident* about — the highest predicted probability
among the wrong answers. These are the interesting failures, because the model was not merely
unsure, it was wrong and sure of it.

When you look at the grid, sort the failures into three buckets:

1. **Genuinely ambiguous handwriting** — even you cannot tell what it is. Nothing to fix; this is
   the irreducible noise floor of MNIST.
2. **Mislabelled ground truth** — MNIST really does contain a handful of these.
3. **Failures an FNN should not make** — e.g. a perfectly clear digit that happens to be written
   larger, thinner, or shifted a few pixels off centre. *These* are the flatten's fault: our model
   has no notion that a shifted digit is the same digit. Spotting one of these is the whole reason
   we did MNIST with an FNN first.

The `all_probs` tensor we saved earlier is what makes this analysis possible — one more argument
for storing predictions rather than just an accuracy number.

In [ ]:
# Which test images did we get wrong?  (Indices line up with test_dataset because
# the test loader used shuffle=False.)
wrong_mask = (all_preds != all_labels)
wrong_indices = wrong_mask.nonzero(as_tuple=True)[0]
print(f"{len(wrong_indices)} of {len(test_dataset)} test images were misclassified "
      f"({100 * len(wrong_indices) / len(test_dataset):.2f}%).\n")

# The model's confidence in its (wrong) answer = the largest softmax probability.
confidence = all_probs.max(dim=1).values

# Sort the mistakes from most to least confident and keep the worst 16.
order = confidence[wrong_indices].argsort(descending=True)
worst = wrong_indices[order][:16]

# Build informative titles: what it said, how sure it was, and what the truth was.
titles = [
    f"said {all_preds[i].item()} ({confidence[i].item():.0%})\ntrue {all_labels[i].item()}"
    for i in worst
]

show_digits(test_dataset, indices=worst.tolist(), titles=titles,
            suptitle="The model's most CONFIDENT mistakes (prediction vs truth)")

# For contrast: the mistakes it was LEAST sure about - usually genuinely ambiguous scrawls.
least = wrong_indices[order][-16:]
titles_least = [
    f"said {all_preds[i].item()} ({confidence[i].item():.0%})\ntrue {all_labels[i].item()}"
    for i in least
]
show_digits(test_dataset, indices=least.tolist(), titles=titles_least,
            suptitle="The model's least confident mistakes (it was unsure - and wrong)")

---
# Step 8 · Summary, Findings, and Exercises

## What we did

| Step | Result |
|---|---|
| 1. Loaded & preprocessed | 70 000 MNIST images, `ToTensor` + `Normalize(0.1307, 0.3081)`, flattened 28×28 → 784 |
| 2. Split | 55 000 train / 5 000 validation (`random_split`) / 10 000 official test; `DataLoader`s with `batch_size=64` |
| 3. Model | `MnistFNN`: 784 → 256 → 128 → 10, ReLU + Dropout(0.2), **235 146** parameters |
| 4. Loss & optimizer | `CrossEntropyLoss` + `Adam(lr=0.001)` — **unchanged from Notebook 01** |
| 5. Trained | 15 epochs × 860 mini-batches = **12 900 weight updates** |
| 6. Evaluated | Test accuracy **98.06%** on the 10 000 unseen official test images (194 errors) |
| 7. Visualised | Loss/accuracy curves, 10×10 confusion matrix, gallery of the worst errors |

## Key findings

1. **Mini-batching is what fixed Notebook 01.** Same loss, same optimizer, same learning rate —
   but 12 900 updates instead of 100. The lesson is that *number of weight updates*, not number of
   epochs, is what drives convergence.
2. **A plain FNN is a strong MNIST baseline.** 98.06% from nothing but `Linear` layers and ReLU.
   Do not reach for a complicated model before you know what a simple one scores.
3. **But the flatten has a real cost.** 784 → 256 alone burns 201 000 parameters — 85% of the
   whole network — and the model still has no concept that neighbouring pixels are related, or
   that a shifted digit is the same digit. That missing 1.2% versus a CNN is more than *half* of
   all remaining errors.
4. **The errors are structured, not random.** 4↔9, 3↔5, 7↔9: the confusion matrix reflects genuine
   shape similarity in handwriting.
5. **Big splits give trustworthy numbers.** Validation (5 000) and test (10 000) accuracy agree to
   a few tenths of a percent — nothing like the 0.87-vs-0.60 whiplash of Notebook 01's 15-sample
   splits.
6. **Overfitting is now visible — as a widening gap, not a rising curve.** After ~epoch 4 the
   training loss keeps falling (≈0.07 → 0.04) while the validation loss stalls and bounces around
   0.08. Dropout is holding back the classic upward turn; remove it (Exercise 7) and it appears.

## Exercises

Work through these in order. Before each experiment, **re-run the model cell (3.2) and the
optimizer cell (Step 4)** so you start from fresh weights and a matching optimizer.

**Understanding the architecture**

1. **Remove the hidden layers.** Replace the model with a single `nn.Linear(784, 10)`. This is
   multinomial logistic regression — no hidden layer, no ReLU. You should land around 92%.
   How many parameters does it have, and how much did the two hidden layers actually buy you?
2. **Ablate the non-linearity.** Keep all three layers but delete both `F.relu` calls. Explain the
   accuracy you get. (Compare it with Exercise 1 — and with what you learned in Notebook 01 about
   stacked linear layers.)
3. **Width sweep.** Try `784 → 64 → 32 → 10` and `784 → 512 → 256 → 10`. Plot test accuracy
   against parameter count. Where do the returns stop being worth it?

**Understanding the training process**

4. **Batch-size sweep.** Train with `batch_size` of 8, 64, and 512 for the same 5 epochs. Record
   updates per epoch, wall-clock time per epoch, and final accuracy. Explain the trade-off you
   observe.
5. **Early stopping.** Track the best validation loss; whenever it improves, save the weights with
   `torch.save(model.state_dict(), 'best.pt')`. After training, reload the best checkpoint with
   `load_state_dict` and evaluate. Does the test accuracy improve over using the final epoch?
6. **Does normalisation matter?** Drop `transforms.Normalize` from the pipeline, keeping only
   `ToTensor()`, and retrain. Compare the first three epochs in particular.
7. **Dropout sweep.** Try `dropout_p=0.0` and `dropout_p=0.5`. With `0.0`, how early does the
   validation loss start rising? This is overfitting, made visible.
8. **Optimizer comparison.** Swap Adam for `optim.SGD(model.parameters(), lr=0.1, momentum=0.9)`.
   Which converges faster in the first epoch, and which ends up better after 15?

**Going deeper**

9. **Show the probabilities.** Pick 8 test images and plot each one next to a bar chart of its 10
   softmax probabilities. Contrast a confidently-correct digit with one of the errors from 7.3.
10. **Test the shift weakness.** Take a correctly-classified test image and shift it 3 pixels
    right with `torch.roll(img, shifts=3, dims=2)`. Does the prediction survive? Try several
    digits, and connect the result back to the flatten discussion in Step 1.6.
11. **Look at what fc1 learned.** Each row of `model.fc1.weight` is 784 numbers — one weight per
    pixel — so `model.fc1.weight[k].view(28, 28)` can be plotted as an image. Show 16 of these
    "templates". Can you spot stroke-like patterns?
12. **Harder dataset, same code.** Change `datasets.MNIST` to `datasets.FashionMNIST` (same shapes,
    same 10-class structure, so *nothing else needs to change*). Accuracy will drop to ~88%. Why
    is clothing harder than digits for a network that cannot see spatial structure?

## Where this goes next

You have now run the 6-step recipe twice: on 4 tabular features, and on 784 image pixels. The
recipe did not change — only the shapes of the tensors and the size of the model.

The one thing left unresolved is the **flatten**. Exercise 10 makes the weakness concrete: shift a
digit slightly and an FNN can fail, because it never knew the pixels had a geometry in the first
place. Fixing that properly requires a layer that looks at small neighbourhoods of pixels and
reuses the same detector across the whole image — which is exactly where the course goes next.